In [ ]:
# =========================================
# 🚀 Setup for Google Colab
# =========================================

# Install SpectroChemPy with CP (TensorLy)
!pip install -q "spectrochempy[cp] @ git+https://github.com/atravert/spectrochempy.git@feature/CP" >& install.log

# Clone the repo
!git clone https://github.com/atravert/MES_demo-operando8.git

# Move into repo
%cd MES_demo-operando8

print("✅ Setup complete")

In [ ]:
## Imports

import numpy as np
import matplotlib.pyplot as plt 
import spectrochempy as scp
import utilities as u

# I. Analysis of Raman MES-Raman data from Hoyer & Hess (2024)

This dataset is described in detail in Hoyer & Hess, *J*. *Phys*. *Chem*. *C* 128, 50, 21377 (**2024**) availabel at [https://doi.org/10.1021/acs.jpcc.4c05826](https://doi.org/10.1021/acs.jpcc.4c05826). It has been made available in Zenodo by their authors at [https://doi.org/10.5281/zenodo.13929684](https://doi.org/10.5281/zenodo.13929684). 

For convenience, we have extracted and copied this dataset in the `/data/Hoyer Hess dataset` folder, and made dedicated reading function in  `utilitiites.py`.

The dataset consists in 40 cycles x 40 spectra per cycle. Let's read and plot it.

In [ ]:
# read and pmlot raw raman data
D_1 = u.read_raman_raw('data/Hoyer Hess dataset/Fig_2-a-c_raw-data.txt')
D_1.y -= D_1.y[0] 
D_1.plot(title='raw spectra')
scp.show()

The spikes, typical of Raman spectroscopy are removed below in a somilar way as in the original paper, by comparing successive spectra.

In [ ]:
# remove spikes
D_2 = D_1.copy()
diff = D_1[1:] - D_1[:-1]
spectrum, spike = np.where(diff.data > 100)
for spec_idx, spike_idx in zip(spectrum, spike):
    D_2.data[spec_idx+1, spike_idx] = 0.5 * (D_1.data[spec_idx+2, spike_idx] + D_1.data[spec_idx, spike_idx])

D_2.plot(title='raw spectra, despiked')
scp.show()

##  Data reshaping

The natural data structure for MES, is a 3way array: (cycle x relative time x wavenumbers).  Below we reshape the dataset accordingly. Note that you can explore the data structuyre by clicking on  the html output of the bext cell which should read something like: ("NDDataset: [float64] u⋅a (shape: (z:41, y:60, x:984))[NDDataset_....]")

In [ ]:
N = 41 # cycles
n = 60 # spectra per period


D_3way = scp.NDDataset(D_2.data.reshape(N, n, D_2.shape[1]))
times = D_2.y.data.reshape(N, n) 
ave_rel_times = np.mean(times - np.expand_dims(times[:,0], axis=1), axis=0)
D_3way.units = D_2.units
D_3way.title = D_2.title
D_3way.set_coordset(z=scp.Coord(np.arange(1, N+1, 1), title='cycle', units=None),
                        y=scp.Coord(ave_rel_times, title='time', units='s'),
                        x= D_2.x)
D_3way

## Data averaging for PSD

PSD requires prior averaging of the spectra. The 3D structure facilitates the process because the `scp.mean()` allows setting the dimension(s) alo,ng which the average is made.  (Note:  the `PSD` class implemented in Spectrochempy also does that autimatically when a 2D dataset is provided, see the documentation). 

In [ ]:
# averaging along the 1st dimension (index 0) and plotting
D_ave = scp.mean(D_3way, dim=0)
D_ave.plot(title='Direct spectra averaged over the cycles')
scp.show()

## MES-PSD

The `PSD` class in spectrochempy requires the input mode and/or number of spectra per cycles to be defined. The `fit()`method carryout the tranform withy default parameters (k=1) and pahse angles. See the documentation for more info. The demodilated spectra ar stored in the `prs` attricute (for Phase Resolved Spectra):

In [ ]:
psd = scp.PSD(n_spectra_per_cycle=60).fit(D_ave)
_ = psd.prs.plot(title='Phase resolved Spectra')

## MES-DAS 

DAS stabnds for Denoise - Average - Subtract and consists in denoising the raw MES spectra. Tbis can be done by Trucated PCA (or equivalently SVD). Note that PCA or SVD  handle 2way arrays so we work on the despiked spectra in `D_2` (despiked, non averaged dataset).

In [ ]:
pca = scp.PCA(n_components=5)
pca.fit(D_2)
D_2_d = pca.inverse_transform()
_ = D_2_d.plot(title='despiked and denoised spectra')

The next step is cycle-averaging -- as already seen above -- followed by subtraction

In [ ]:
# reshape
D_3way_d = scp.NDDataset(D_2_d.data.reshape(N, n, D_2_d.shape[1]))
D_3way_d.units = D_2_d.units
D_3way_d.title = D_2_d.title
D_3way_d.set_coordset(z=scp.Coord(np.arange(1, N+1, 1), title='cycle', units=None),
                        y=scp.Coord(ave_rel_times, title='time', units='s'),
                        x= D_2_d.x)

# avearage along the 1st dim
D_d_ave = scp.mean(D_3way_d, dim=0)

# subtratct the 1st spectrum (index O)
D_d_ave_dif = D_d_ave - D_d_ave[0] 

# compute also the differene spectra withoiut denoising  
D_ave_dif = D_ave - D_ave[0]

# Now compare plots in the 200-800  cm-1 reguion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

D_ave_dif.plot(ax=axes[0], xlim=(300, 800), title='Averaged differene spectra, no denoising')
D_d_ave_dif.plot(ax=axes[1], xlim=(300, 800), title='Averaged differene spectra with prior denoising')

plt.tight_layout()
plt.show()

A key advantage of spectra in the time domain is the possibility to determine the kinetic profile(s). In thuis cases there is a single reaction so all spectra are equal excpept for noise.
The most straightformward way is to integrate a peak belonging to one of the reactant(s) or product(s). 


In [ ]:
C = D_d_ave_dif[:, 500.:600.].trapezoid() 
C = C /  np.max(C)
_ = C.plot(title='Scaled extent of reaction')

## CP (Candecomp/Parafac) decomposition

In general, the most reliable results are obtained when CP is carried out on spectra subtracted from the 1st averaged spectrum.

In [ ]:
# Fist subtract from each cycle the mean first spectrum. 
D_3way_diff = D_3way - D_3way[:,0,:].data.mean(axis=0)  

It is also important co carry out the CP for an ivreasing number of components and check the Core Consoistency criterion. 

In [ ]:
cp_list = []

for n_components in [1,2]:
    cp = scp.CP(n_components=n_components)
    cp.fit(D_3way_diff)
    print(f'{n_components} components: CC = {cp.core_consistency} ')
    cp_list.append(cp)

The core consistency diganostic suggests a single component CP. The loadings are plotted below. NB: the A, B and C matrices can be arbitrarily multiplied by three numbers a, b, c with abc=1 ; below we use a = -1, b= 1, c=-1 so as to have consistent profiles. 

In [ ]:
cp  = cp_list[0]
loadings = cp.loadings

A = - cp.A
B = cp.B
C = - cp.C


# plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

A.plot(ax=axes[0], ylim=(0, 3000))
B.plot(ax=axes[1], ylim=(0, 0.25))
C.plot(ax=axes[2])

axes[0].set_title("A")
axes[1].set_title("B")
axes[2].set_title("C")

plt.tight_layout()
plt.show()


The A matrix suggestst reflects the intensitry evolution between the cycles. The B matrix reflects the mean evoution od spectral intensity with a cycle, and C the averaged difference spectral profile. Both are very consistent with the concentration profile derived from the mean diference spectra (cf above)      

# II. Analysis of Raman MES-Raman data from De Coster et al (2024)

This methodology is described in detail in De Coster et al. *ACS Catal*. 13, 5084–5095 (**2023**) available at [https://doi.org/10.1021/acscatal.3c00646](https://doi.org/10.1021/acscatal.3c00646). The dataset has been made available in Zenodo by their authors at [https://doi.org/10.5281/zenodo.7780501](https://doi.org/10.5281/zenodo.7780501). 

For convenience, we have extracted and copied this dataset in the `/data/De Coster dataset` folder, and made dedicated reading function in  `utilitiites.py`.

Let's read it and select the last 21 cycles of 24 spectra/ycle (quasi periodic state) and 7100.-7200. eV range and plot it.

In [ ]:
# read normalized XAS, adjust time axis

D = u.read_xas('data/De Coster dataset')
D.y = D.y - D[0].y
D = D[-21*24:, 7100.:7200.]   # select the last 21 cycles and 7100.-7200. eV range  
D.description = "all spectra Fe-Ni-MgFeAlO4_MEXAS_normalized"

_ = D.plot(title='raw spectra')

 ## Reshape, average, and compute PSD 

These steps are the same as for the previous dataset.

In [ ]:

# Reshaping
N = 21 # cycles
n = 24 # spectra per period


D_3way = scp.NDDataset(D.data.reshape(N, n, D.shape[1]))
times = D.y.data.reshape(N, n) 
ave_rel_times = np.mean(times - np.expand_dims(times[:,0], axis=1), axis=0)
D_3way.units = D.units
D_3way.title = D.title
D_3way.set_coordset(z=scp.Coord(np.arange(1, N+1, 1), title='cycle', units=None),
                        y=scp.Coord(ave_rel_times, title='time', units='s'),
                        x= D.x)

# Averaging
D_ave = scp.mean(D_3way, dim=0)

# Difference spectra
D_ave_d = D_ave  - D_ave[0] 

# PSD
psd = scp.PSD(n_spectra_per_cycle=24)
psd.fit(D_ave_d)

# plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

D_ave.plot(ax=axes[0], title='Averaged spectra')
D_ave_d.plot(ax=axes[1], title='Averaged difference  spectra')
psd.prs.plot(ax=axes[2], title='demodulated  spectra, k=1')


plt.tight_layout()
plt.show()


## CP (Candecomp Parafac) decomposion.

The methodology as the same as previopusmly. Here the difference spectra suggest at least 2 contributions so we carru=y pout the CP for 1, 2 and 3 components 

In [ ]:
# Fist subtract from each cycle the mean first spectrum. 
D_3way_diff = D_3way - D_3way[:,0,:].data.mean(axis=0)  

# Then Carry out CPs
cp_list = []

for n_components in [1,2,3]:
    cp = scp.CP(n_components=n_components)
    cp.fit(D_3way_diff)
    print(f'{n_components} components: CC = {cp.core_consistency} ')
    cp_list.append(cp)

The CC diagnostic also suggests 2 components. 

In [ ]:
cp  = cp_list[1]
loadings = cp.loadings

A = cp.A
B = cp.B
C = cp.C


# plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Note that depending on the initial shape of the 3way array, the dimensions of 
# the loadings may have to be transposed before plotting, as here. This may be 
# changed in a coming release of SpectroChemmPy.

A.T.plot(ax=axes[0])
B.T.plot(ax=axes[1])
C.T.plot(ax=axes[2])

axes[0].set_title("A")
axes[1].set_title("B")
axes[2].set_title("C")

plt.tight_layout()
plt.show()

In line with time domain data, CP provides consistent spectral and kinetc profiles of successiove reactions. Note that the profiles related to the first reaction are particularly noisy, probably because of its low contributioon to overall signal. 